In [ ]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())


# Molecular Docking with AutoDock Vina
## A Hands-on Workshop — Virtual Screening of 117 Ligands against Trypsin

**Author:** Omar Hamouda 

 **Environment:** Python 3.11 | VS Code (WSL/Ubuntu) | Anaconda

---

## 📌 Overview

This notebook covers **molecular docking** — the computational prediction of how small molecules (ligands) bind to a protein's active site.

We will:
1. **Prepare** the receptor (Trypsin, `2ZQ2`) in PDBQT format.
2. **Prepare** all 117 ligands from Lesson 13 in PDBQT format.
3. **Run docking** with **smina** (AutoDock Vina fork) for each ligand.
4. **Rank** ligands by **binding affinity** (kcal/mol).
5. **Analyze** the top hits with ProLIF.

---

## 📚 Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | Setup & Imports | Libraries and tools |
| 2 | Receptor Preparation | Convert protein to PDBQT |
| 3 | Binding Box Definition | Define the search space |
| 4 | Ligand Preparation | Convert 117 SDF files to PDBQT |
| 5 | Run Docking | Execute smina for each ligand |
| 6 | Analyze Results | Rank by binding affinity |
| 7 | ProLIF on Top Hits | Interaction fingerprints |
| 8 | Summary & Next Steps | Recap |

---

## 📁 Data Files

**Receptor:**
- `pdb/protein_h.pdb` — Hydrogenated Trypsin (from Lesson 14)

**Ligands:**
- `ligands/*.sdf` — 117 ligands from Lesson 13

**Reference:**
- `2ZQ2` — Crystal structure with ligand 13U bound

In [ ]:
# Verify all required libraries
import os
import subprocess
from rdkit import Chem
import prolif as plf
import MDAnalysis as mda

print("✅ All imports successful.")
print(f"   RDKit loaded")
print(f"   ProLIF: {plf.__version__}")
print(f"   MDAnalysis: {mda.__version__}")

In [ ]:
# Verify smina is installed and accessible
result = subprocess.run(
    ["smina", "--version"],
    capture_output=True,
    text=True
)
print(result.stdout)

## 1️⃣ Setup & Imports

In [ ]:
# ─── Standard Library ───
import os
import subprocess

# ─── Cheminformatics ───
from rdkit import Chem

# ─── Structure Processing ───
import MDAnalysis as mda
import prolif as plf

# ─── Set up directories ───
os.makedirs("docking", exist_ok=True)
os.makedirs("docking/receptor", exist_ok=True)
os.makedirs("docking/ligands", exist_ok=True)
os.makedirs("docking/results", exist_ok=True)

print("✅ Setup complete. Directories created:")
print("   docking/receptor/   → protein PDBQT")
print("   docking/ligands/    → ligand PDBQTs")
print("   docking/results/    → docking outputs")

### 📝 Section Summary

In this section, we:
- **Imported** all required libraries for docking.
- **Understood** the docking concept (receptor + ligand → affinity).
- **Created** the folder structure for docking files.

> **Key takeaway:** Docking requires **clean inputs**: receptor in PDBQT, ligand in PDBQT, and a defined binding box.

## 2️⃣ Receptor Preparation

In [ ]:
# ─── Locate the hydrogenated protein from Lesson 14 ───
protein_pdb = "pdb/protein_h.pdb"

if os.path.exists(protein_pdb):
    size = os.path.getsize(protein_pdb)
    print(f"✅ Found: {protein_pdb}")
    print(f"   Size: {size:,} bytes")
    
    # Count atoms
    with open(protein_pdb, "r") as f:
        atoms = [line for line in f if line.startswith(("ATOM", "HETATM"))]
    print(f"   Total atoms: {len(atoms):,}")
else:
    print(f"❌ NOT found: {protein_pdb}")
    print("   Run Lesson 14 first to create it.")

In [ ]:
# ─── Convert protein PDB → PDBQT using OpenBabel ───
receptor_pdbqt = "docking/receptor/2zq2_receptor.pdbqt"

cmd = [
    "obabel",
    "-ipdb", protein_pdb,
    "-opdbqt",
    "-O", receptor_pdbqt,
    "-xr"   # rigid receptor (no flexible residues)
]

result = subprocess.run(cmd, capture_output=True, text=True)

print(f"Return code: {result.returncode}")
print(f"stdout: {result.stdout}")
if result.stderr:
    print(f"stderr: {result.stderr[:500]}")

if os.path.exists(receptor_pdbqt):
    size = os.path.getsize(receptor_pdbqt)
    print(f"\n✅ Receptor PDBQT created: {receptor_pdbqt}")
    print(f"   Size: {size:,} bytes")

In [ ]:
# ─── Inspect the first few lines of the PDBQT ───
with open(receptor_pdbqt, "r") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines):,}\n")
print("First 10 lines:")
print("─" * 60)
for line in lines[:10]:
    print(line.rstrip())
print("─" * 60)

### 📝 Section Summary

In this section, we:
- **Located** the hydrogenated protein from Lesson 14.
- **Converted** it to PDBQT format using OpenBabel (`-xr` for rigid).
- **Verified** the output (3,220 atoms).

> **Key takeaway:** PDBQT is the **required format** for Vina/smina. OpenBabel handles the conversion, adding partial charges and AutoDock atom types.

## 3️⃣ Binding Box Definition


In [ ]:
# ─── Extract the center of mass of the original ligand (13U) ───
# This will be the center of our docking box

# Load the original 2zq2 structure
u_original = mda.Universe("pdb/2zq2.pdb")

# Select the original ligand
ligand_original = u_original.select_atoms("resname 13U and altLoc A")

print(f"Original ligand atoms: {len(ligand_original)}")

# Compute center of mass
center = ligand_original.center_of_mass()
print(f"\nBox Center (Å):")
print(f"   x = {center[0]:.2f}")
print(f"   y = {center[1]:.2f}")
print(f"   z = {center[2]:.2f}")

In [ ]:
# ─── Define the binding box dimensions ───
box_size = 25.0  # Å per side — standard for drug-like ligands

# Compute the box bounds (min/max) for each axis
half = box_size / 2

box_min = [center[i] - half for i in range(3)]
box_max = [center[i] + half for i in range(3)]

print(f"Box size: {box_size} × {box_size} × {box_size} Å\n")
print("Box bounds (Å):")
print(f"   x: [{box_min[0]:7.2f}, {box_max[0]:7.2f}]")
print(f"   y: [{box_min[1]:7.2f}, {box_max[1]:7.2f}]")
print(f"   z: [{box_min[2]:7.2f}, {box_max[2]:7.2f}]")

In [ ]:
# ─── Save the box configuration for reference ───
config_path = "docking/box_config.txt"

with open(config_path, "w") as f:
    f.write(f"# Docking Box Configuration for 2ZQ2 (Trypsin)\n")
    f.write(f"# Center (Å): {center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f}\n")
    f.write(f"# Size (Å): {box_size} x {box_size} x {box_size}\n")
    f.write(f"\n")
    f.write(f"center_x = {center[0]:.3f}\n")
    f.write(f"center_y = {center[1]:.3f}\n")
    f.write(f"center_z = {center[2]:.3f}\n")
    f.write(f"\n")
    f.write(f"size_x = {box_size}\n")
    f.write(f"size_y = {box_size}\n")
    f.write(f"size_z = {box_size}\n")

print(f"✅ Saved: {config_path}")
print(f"\nConfig content:")
print("─" * 40)
with open(config_path, "r") as f:
    print(f.read())

### 📝 Section Summary

In this section, we:
- **Extracted** the center of mass of the original ligand `13U`.
- **Defined** a 25 × 25 × 25 Å binding box around that center.
- **Saved** the configuration to `docking/box_config.txt`.

> **Key takeaway:** The binding box tells smina **where to search**. A box centered on the known ligand position ensures we explore the correct active site.

## 4️⃣ Ligand Preparation

In [ ]:
# ─── List all ligand SDF files ───
ligands_dir = "ligands"
sdf_files = sorted([
    f for f in os.listdir(ligands_dir)
    if f.endswith(".sdf")
])

print(f"Total SDF files: {len(sdf_files)}")
print(f"\nFirst 10:")
for f in sdf_files[:10]:
    print(f"   {f}")
print(f"   ... and {len(sdf_files) - 10} more.")

In [ ]:
# ─── Convert all SDF → PDBQT using OpenBabel ───
# (Meeko is better but more complex; OpenBabel works for most cases)

ligands_pdbqt_dir = "docking/ligands"

success = []
failed  = []

print("Converting ligands...\n")

for i, sdf_file in enumerate(sdf_files, start=1):
    ligand_id = sdf_file.replace(".sdf", "")
    sdf_path  = os.path.join(ligands_dir, sdf_file)
    pdbqt_path = os.path.join(ligands_pdbqt_dir, f"{ligand_id}.pdbqt")
    
    # Skip if already converted
    if os.path.exists(pdbqt_path):
        success.append(ligand_id)
        continue
    
    cmd = [
        "obabel",
        "-isdf", sdf_path,
        "-opdbqt",
        "-O", pdbqt_path,
        "-h",        # add hydrogens
        "--partialcharge", "gasteiger"
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0 and os.path.exists(pdbqt_path):
        success.append(ligand_id)
    else:
        failed.append((ligand_id, result.stderr[:200]))
    
    if i % 20 == 0:
        print(f"   Progress: {i}/{len(sdf_files)}...")

print(f"\n{'='*50}")
print(f"  ✅ Success: {len(success)}")
print(f"  ❌ Failed:  {len(failed)}")
print(f"{'='*50}")

In [ ]:
# ─── Show failed ligands (if any) ───
if failed:
    print(f"Failed conversions ({len(failed)}):\n")
    for ligand_id, err in failed[:10]:
        print(f"   ❌ {ligand_id}")
        print(f"      {err[:150]}")
else:
    print("✅ All ligands converted successfully!")

In [ ]:
# ─── Verify the converted files ───
pdbqt_files = sorted([
    f for f in os.listdir(ligands_pdbqt_dir)
    if f.endswith(".pdbqt")
])

print(f"Total PDBQT files: {len(pdbqt_files)}\n")
print("First 10:")
for f in pdbqt_files[:10]:
    size = os.path.getsize(os.path.join(ligands_pdbqt_dir, f))
    print(f"   {f}  ({size:,} bytes)")

print(f"\n... and {len(pdbqt_files) - 10} more.")

### 📝 Section Summary

In this section, we:
- **Listed** all 117 SDF ligand files from Lesson 13.
- **Converted** them to PDBQT format using OpenBabel.
- **Added** hydrogens and Gasteiger partial charges.
- **Verified** the output files.

> **Key takeaway:** Ligand preparation is often **the bottleneck** in virtual screening. OpenBabel handles most cases; for challenging molecules, use Meeko.

## 5️⃣ Run Docking

In [ ]:

# ─── Helper function: run docking for a single ligand ───
import time

def run_docking(ligand_id, exhaustiveness=4, num_modes=1):
    """
    Run smina for a single ligand.
    
    Parameters
    ----------
    ligand_id : str
        Ligand 3-letter code (e.g., '13U').
    exhaustiveness : int
        Search thoroughness (higher = slower but better).
    num_modes : int
        Number of binding poses to return.
    
    Returns
    -------
    dict : {'ligand_id', 'affinity', 'time_sec', 'status'}
    """
    receptor = "docking/receptor/2zq2_receptor.pdbqt"
    ligand   = f"docking/ligands/{ligand_id}.pdbqt"
    output   = f"docking/results/{ligand_id}_out.pdbqt"
    
    if not os.path.exists(ligand):
        return {"ligand_id": ligand_id, "affinity": None, "time_sec": 0, "status": "ligand_missing"}
    
    cmd = [
        "smina",
        "--receptor", receptor,
        "--ligand", ligand,
        "--center_x", str(center[0]),
        "--center_y", str(center[1]),
        "--center_z", str(center[2]),
        "--size_x", str(box_size),
        "--size_y", str(box_size),
        "--size_z", str(box_size),
        "--exhaustiveness", str(exhaustiveness),
        "--num_modes", str(num_modes),
        "--out", output
    ]
    
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    t1 = time.time()
    
    # Parse the affinity from stdout
    affinity = None
    for line in result.stdout.split("\n"):
        if line.strip().startswith("Affinity:"):
            try:
                affinity = float(line.split()[1])
            except (ValueError, IndexError):
                pass
            break
    
    return {
        "ligand_id": ligand_id,
        "affinity": affinity,
        "time_sec": round(t1 - t0, 1),
        "status": "ok" if affinity is not None else "no_affinity"
    }

print("✅ Function 'run_docking' defined.")

In [ ]:
# ─── Test with ligand 13U (the reference ligand) ───
print("Testing with ligand 13U...\n")

result_13U = run_docking("13U", exhaustiveness=4)

print(f"Ligand:     {result_13U['ligand_id']}")
print(f"Affinity:   {result_13U['affinity']} kcal/mol")
print(f"Time:       {result_13U['time_sec']} seconds")
print(f"Status:     {result_13U['status']}")

### ⏱️ Time Estimation

Based on the single-ligand test:

| Metric | Value |
|--------|-------|
| **Time per ligand** | ~45 seconds |
| **Total for 117** | ~88 minutes (1.5 hours) |

**Decision:** We can run the **first 20 ligands** as a batch, then decide whether to continue.

> **Tip:** You can reduce `exhaustiveness` to `2` for faster (but less accurate) results.

In [ ]:
# ─── Run docking for the first 20 ligands ───
# (Start small to validate, then run the rest)

sample_ligands = [
    f.replace(".pdbqt", "") 
    for f in sorted(os.listdir("docking/ligands"))
    if f.endswith(".pdbqt")
][:20]

print(f"Running docking for {len(sample_ligands)} ligands...\n")
print(f"Estimated time: ~{len(sample_ligands) * 0.75:.0f} minutes.\n")

results = []
t_start = time.time()

for i, ligand_id in enumerate(sample_ligands, start=1):
    result = run_docking(ligand_id, exhaustiveness=4)
    results.append(result)
    print(f"  [{i:2d}/{len(sample_ligands)}] {ligand_id}: "
          f"affinity = {result['affinity']} kcal/mol "
          f"({result['time_sec']}s)")

t_end = time.time()
print(f"\n{'='*60}")
print(f"  ✅ Completed: {len(results)}")
print(f"  ⏱️  Total time: {(t_end - t_start)/60:.1f} minutes")
print(f"{'='*60}")

In [ ]:
# ─── Sort results by binding affinity ───
import pandas as pd

# Convert to DataFrame
df_results = pd.DataFrame(results)

# Remove ligands that failed
df_valid = df_results[df_results["affinity"].notna()].copy()

# Sort by affinity (most negative = best)
df_valid = df_valid.sort_values("affinity", ascending=True).reset_index(drop=True)

# Add rank
df_valid.insert(0, "rank", range(1, len(df_valid) + 1))

print(f"Top 10 ligands:\n")
print(df_valid[["rank", "ligand_id", "affinity", "time_sec"]].head(10).to_string(index=False))

In [ ]:
# ─── Save results to CSV ───
output_csv = "docking/results/affinities.csv"
df_valid.to_csv(output_csv, index=False)

print(f"✅ Saved: {output_csv}")
print(f"   Rows: {len(df_valid)}")

### 📝 Section Summary

In this section, we:
- **Defined** a function `run_docking()` for running smina on a single ligand.
- **Tested** with ligand `13U` (~45 seconds per run).
- **Ran** docking on 20 ligands as a batch.
- **Sorted** results by binding affinity (most negative = best).
- **Saved** results to `docking/results/affinities.csv`.

> **Key takeaway:** Testing with **one ligand first** is essential. Once the pipeline is validated, batch processing is straightforward.

## 6️⃣ Analyze Results

In [ ]:
# ─── Basic statistics of the docking results ───
print("Docking Results Summary")
print("=" * 60)
print(f"Total ligands docked:   {len(df_valid)}")
print(f"Best affinity:          {df_valid['affinity'].min():.2f} kcal/mol")
print(f"Worst affinity:         {df_valid['affinity'].max():.2f} kcal/mol")
print(f"Mean affinity:          {df_valid['affinity'].mean():.2f} kcal/mol")
print(f"Median affinity:        {df_valid['affinity'].median():.2f} kcal/mol")
print(f"Std deviation:          {df_valid['affinity'].std():.2f} kcal/mol")
print("=" * 60)

# Reference ligand score
ref_affinity = df_valid[df_valid["ligand_id"] == "13U"]["affinity"].values
if len(ref_affinity) > 0:
    print(f"\nReference ligand (13U): {ref_affinity[0]:.2f} kcal/mol")
    better = df_valid[df_valid["affinity"] < ref_affinity[0]]
    print(f"Ligands better than 13U: {len(better)}")

In [ ]:
# ─── Full ranking table ───
print("Full Ranking (20 ligands):\n")
print(df_valid[["rank", "ligand_id", "affinity", "time_sec"]].to_string(index=False))

In [ ]:
# ─── Bar chart of binding affinities ───
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

# Color: reference in red, others in blue
colors = ["red" if lid == "13U" else "steelblue" 
          for lid in df_valid["ligand_id"]]

ax.bar(df_valid["ligand_id"], df_valid["affinity"], color=colors)
ax.set_xlabel("Ligand ID", fontsize=12)
ax.set_ylabel("Binding Affinity (kcal/mol)", fontsize=12)
ax.set_title("Docking Results — Trypsin (2ZQ2)", fontsize=14)
ax.axhline(y=ref_affinity[0] if len(ref_affinity) else -8.5,
           color="red", linestyle="--", alpha=0.5,
           label=f"Reference (13U)")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("docking/results/affinity_chart.png", dpi=150)
plt.show()

print("✅ Chart saved: docking/results/affinity_chart.png")

In [ ]:
# ─── Select top 5 ligands for further analysis ───
top_5 = df_valid.head(5)["ligand_id"].tolist()

print("Top 5 ligands selected for ProLIF analysis:\n")
for i, lid in enumerate(top_5, start=1):
    aff = df_valid[df_valid["ligand_id"] == lid]["affinity"].values[0]
    print(f"   {i}. {lid}   ({aff:.2f} kcal/mol)")

# Save to file
with open("docking/results/top_5.txt", "w") as f:
    for lid in top_5:
        f.write(f"{lid}\n")

print(f"\n✅ Saved: docking/results/top_5.txt")

### 📝 Section Summary

In this section, we:
- **Analyzed** the affinity distribution (min, max, mean).
- **Compared** each ligand to the reference `13U`.
- **Visualized** results with a bar chart.
- **Selected** the top 5 ligands for ProLIF analysis.

### 📊 Results Interpretation

| Observation | Meaning |
|-------------|---------|
| **Narrow range (-5 to -9)** | All ligands bind reasonably well |
| **13U ranks high** | Reference ligand is a strong binder |
| **Top hits are near -8.5** | Comparable to known inhibitor |

> **Key takeaway:** Docking is a **ranking tool**. The absolute values matter less than the **relative ordering**.

## 6.5️⃣ Full Batch — Remaining 97 Ligands

In [ ]:
# ─── Identify remaining ligands ───
all_ligands = [
    f.replace(".pdbqt", "") 
    for f in sorted(os.listdir("docking/ligands"))
    if f.endswith(".pdbqt")
]

already_done = set(df_valid["ligand_id"])
remaining = [lig for lig in all_ligands if lig not in already_done]

print(f"Already docked: {len(already_done)}")
print(f"Remaining:      {len(remaining)}")
print(f"Total:          {len(all_ligands)}\n")

if len(remaining) == 0:
    print("✅ All ligands already docked!")
else:
    print(f"⏱️  Estimated time: ~{len(remaining) * 0.75:.0f} minutes\n")
    print("Starting full batch docking...")
    print("(This will take a while — go get coffee ☕)\n")

### ⚠️ Technical Note

An earlier version of this cell ran a full batch loop, but failed:
ValueError: cannot insert rank, already exists

**Cause:** The `rank` column already existed in `df_valid`, causing a conflict during `pd.concat`.

**Fix:** We recovered all results directly from the **output PDBQT files** using a dedicated parser — no re-run required.

> **Lesson:** When a batch job fails, check the output files first. Data is often recoverable.

In [ ]:
# Check if results are still in memory
if "new_results" in dir():
    print(f"✅ new_results is in memory: {len(new_results)} items")
else:
    print("❌ new_results is NOT in memory")
    
if "df_all" in dir():
    print(f"✅ df_all is in memory: {len(df_all)} rows")
else:
    print("❌ df_all is NOT in memory")
    

In [ ]:
import os
import pandas as pd

csv_path = "docking/results/affinities_full.csv"

if os.path.exists(csv_path):
    print(f"✅ File exists: {csv_path}")
    print(f"   Size: {os.path.getsize(csv_path):,} bytes")
    
    # Load it back
    df_all = pd.read_csv(csv_path)
    print(f"   Rows: {len(df_all)}")
    print(f"   Columns: {list(df_all.columns)}")
    print()
    print("First 5 rows:")
    print(df_all.head())
else:
    print(f"❌ File not found: {csv_path}")
    print("   You need to re-run the docking analysis.")

In [ ]:
# Re-rank and save
if "rank" in df_all.columns:
    df_all = df_all.drop(columns=["rank"])

df_all.insert(0, "rank", range(1, len(df_all) + 1))
df_all = df_all.sort_values("affinity", ascending=True).reset_index(drop=True)

df_all.to_csv("docking/results/affinities_full.csv", index=False)

print(f"✅ Re-saved: docking/results/affinities_full.csv")
print(f"   Total ligands: {len(df_all)}")
print()
print("Top 10 ligands:")
print(df_all.head(10))

In [ ]:
# ─── Extract affinities from all output files ───
import glob

results_dir = "docking/results"
output_files = sorted(glob.glob(f"{results_dir}/*_out.pdbqt"))

print(f"Found {len(output_files)} output files\n")

recovered = []
for f in output_files:
    ligand_id = os.path.basename(f).replace("_out.pdbqt", "")
    
    # Parse affinity from the output file (smina writes REMARK VINA RESULT)
    affinity = None
    with open(f, "r") as fh:
        for line in fh:
            if "REMARK VINA RESULT:" in line or "REMARK minimizedAffinity" in line:
                try:
                    # Format: "REMARK VINA RESULT:    -8.5   0.000   0.000"
                    parts = line.split()
                    for p in parts:
                        try:
                            val = float(p)
                            if val < 0:
                                affinity = val
                                break
                        except ValueError:
                            continue
                except Exception:
                    pass
                if affinity:
                    break
    
    if affinity is not None:
        recovered.append({
            "ligand_id": ligand_id,
            "affinity": affinity,
        })

print(f"Recovered: {len(recovered)} affinities\n")

# Build sorted DataFrame
df_all = pd.DataFrame(recovered)
df_all = df_all.sort_values("affinity", ascending=True).reset_index(drop=True)
df_all.insert(0, "rank", range(1, len(df_all) + 1))

# Save
df_all.to_csv(f"{results_dir}/affinities_full.csv", index=False)
print(f"✅ Saved: {results_dir}/affinities_full.csv")
print(f"   Total ligands: {len(df_all)}\n")

# Show top 10
print("Top 10 ligands:")
print(df_all.head(10).to_string(index=False))

In [ ]:
# ─── Investigate the 18 missing ligands ───
all_output = set(
    os.path.basename(f).replace("_out.pdbqt", "")
    for f in glob.glob("docking/results/*_out.pdbqt")
)
recovered_ids = set(df_all["ligand_id"])
missing_ids = sorted(all_output - recovered_ids)

print(f"Missing ligands: {len(missing_ids)}\n")
print("Missing list:")
for lid in missing_ids:
    # Check file size
    path = f"docking/results/{lid}_out.pdbqt"
    size = os.path.getsize(path)
    
    # Read first few lines
    with open(path, "r") as f:
        first_lines = [f.readline().strip() for _ in range(3)]
    
    print(f"  {lid}  ({size} bytes)")
    for line in first_lines:
        if line:
            print(f"      {line}")

In [ ]:
# ─── Investigate the 18 missing ligands ───
import glob

all_output = set(
    os.path.basename(f).replace("_out.pdbqt", "")
    for f in glob.glob("docking/results/*_out.pdbqt")
)
recovered_ids = set(df_all["ligand_id"])
missing_ids = sorted(all_output - recovered_ids)

print(f"Missing ligands: {len(missing_ids)}\n")
print("Missing list:")
for lid in missing_ids:
    path = f"docking/results/{lid}_out.pdbqt"
    size = os.path.getsize(path)
    
    with open(path, "r") as f:
        first_lines = [f.readline().strip() for _ in range(3)]
    
    print(f"  {lid}  ({size} bytes)")
    for line in first_lines:
        if line:
            print(f"      {line}")

In [ ]:
# ─── Check the source PDBQT files for the missing ligands ───
import glob
import subprocess

missing_ids = ['0KV', '0ZW', '0ZX', '0ZY', '132', '334', '678', '762', '847', '907',
               '972', '991', 'BAZ', 'BOZ', 'BR6', 'BRV', 'PPB', 'ZEN']

receptor = "docking/receptor/2zq2_receptor.pdbqt"

print(f"Checking {len(missing_ids)} ligands...\n")

for lid in missing_ids[:5]:  # Show first 5
    pdbqt = f"docking/ligands/{lid}.pdbqt"
    
    if not os.path.exists(pdbqt):
        print(f"  {lid}: ❌ PDBQT missing")
        continue
    
    size = os.path.getsize(pdbqt)
    print(f"  {lid}: ({size} bytes)")
    
    # Run smina manually and capture output
    cmd = [
        "smina",
        "--receptor", receptor,
        "--ligand", pdbqt,
        "--center_x", str(center[0]),
        "--center_y", str(center[1]),
        "--center_z", str(center[2]),
        "--size_x", str(box_size),
        "--size_y", str(box_size),
        "--size_z", str(box_size),
        "--exhaustiveness", "1",
        "--num_modes", "1"
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    print(f"    Return code: {result.returncode}")
    if result.stderr:
        print(f"    STDERR: {result.stderr[:200]}")
    print()

In [ ]:
# ─── Classify missing ligands into two categories ───
missing_ids = ['0KV', '0ZW', '0ZX', '0ZY', '132', '334', '678', '762', '847', '907',
               '972', '991', 'BAZ', 'BOZ', 'BR6', 'BRV', 'PPB', 'ZEN']

empty_pdbqt = []      # PDBQT has no atoms
valid_pdbqt = []      # PDBQT is fine, docking failed

for lid in missing_ids:
    pdbqt = f"docking/ligands/{lid}.pdbqt"
    if not os.path.exists(pdbqt):
        empty_pdbqt.append(lid)
        continue
    
    # Check atom count
    with open(pdbqt) as f:
        atoms = [l for l in f if l.startswith(("ATOM", "HETATM"))]
    
    if len(atoms) < 5:  # Suspiciously few
        empty_pdbqt.append(lid)
    else:
        valid_pdbqt.append(lid)

print(f"Empty PDBQT ({len(empty_pdbqt)}):")
for lid in empty_pdbqt:
    print(f"   {lid}")

print(f"\nValid PDBQT ({len(valid_pdbqt)}):")
for lid in valid_pdbqt:
    print(f"   {lid}")

In [ ]:
# ─── Re-run docking for the valid PDBQTs that failed ───
print("Re-running docking for the valid PDBQTs...\n")

for lid in valid_pdbqt:
    # Delete the empty output
    out_path = f"docking/results/{lid}_out.pdbqt"
    if os.path.exists(out_path):
        os.remove(out_path)
    
    # Re-run
    result = run_docking(lid, exhaustiveness=4)
    print(f"  {lid}: affinity = {result['affinity']} kcal/mol")

In [ ]:
# ─── FIXED version of run_docking ───
import time

def run_docking_v2(ligand_id, exhaustiveness=4, num_modes=1):
    """
    Fixed version — parses affinity from the OUTPUT FILE (not stdout).
    """
    receptor = "docking/receptor/2zq2_receptor.pdbqt"
    ligand   = f"docking/ligands/{ligand_id}.pdbqt"
    output   = f"docking/results/{ligand_id}_out.pdbqt"
    
    if not os.path.exists(ligand):
        return {"ligand_id": ligand_id, "affinity": None, "status": "ligand_missing"}
    
    # Check ligand has atoms
    with open(ligand) as f:
        atoms = [l for l in f if l.startswith(("ATOM", "HETATM"))]
    if len(atoms) < 5:
        return {"ligand_id": ligand_id, "affinity": None, "status": "empty_ligand"}
    
    # Delete stale output
    if os.path.exists(output):
        os.remove(output)
    
    cmd = [
        "smina",
        "--receptor", receptor,
        "--ligand", ligand,
        "--center_x", str(center[0]),
        "--center_y", str(center[1]),
        "--center_z", str(center[2]),
        "--size_x", str(box_size),
        "--size_y", str(box_size),
        "--size_z", str(box_size),
        "--exhaustiveness", str(exhaustiveness),
        "--num_modes", str(num_modes),
        "--out", output
    ]
    
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    t1 = time.time()
    
    # ─── Parse affinity from OUTPUT FILE ───
    affinity = None
    if os.path.exists(output):
        with open(output) as f:
            for line in f:
                if "REMARK VINA RESULT:" in line:
                    try:
                        # Format: "REMARK VINA RESULT:  -8.5  0.000  0.000"
                        parts = line.split()
                        affinity = float(parts[3])
                    except (ValueError, IndexError):
                        pass
                    break
    
    return {
        "ligand_id": ligand_id,
        "affinity": affinity,
        "time_sec": round(t1 - t0, 1),
        "status": "ok" if affinity is not None else "no_affinity"
    }

print("✅ Function 'run_docking_v2' defined.")

In [ ]:
# ─── Re-run for valid PDBQTs using the FIXED function ───
print("Re-running docking for valid PDBQTs...\n")

rerun_results = []
for lid in valid_pdbqt:
    result = run_docking_v2(lid, exhaustiveness=4)
    rerun_results.append(result)
    print(f"  {lid}: affinity = {result['affinity']} kcal/mol ({result['time_sec']}s)")

print(f"\n✅ Re-run complete: {len(rerun_results)}")

In [ ]:
# ─── Inspect the actual output file ───
test_file = "docking/results/0KV_out.pdbqt"

if os.path.exists(test_file):
    print(f"File size: {os.path.getsize(test_file)} bytes\n")
    
    with open(test_file) as f:
        lines = f.readlines()
    
    print(f"Total lines: {len(lines)}\n")
    print("All REMARK lines:")
    print("─" * 60)
    for line in lines:
        if "REMARK" in line.upper() or "Affinity" in line or "affinity" in line:
            print(line.rstrip())
    print("─" * 60)
else:
    print(f"❌ File not found: {test_file}")

In [ ]:
# ─── FINAL version — parses 'minimizedAffinity' (smina format) ───
import time

def run_docking_final(ligand_id, exhaustiveness=4, num_modes=1):
    """
    Run smina and parse 'minimizedAffinity' from the output file.
    """
    receptor = "docking/receptor/2zq2_receptor.pdbqt"
    ligand   = f"docking/ligands/{ligand_id}.pdbqt"
    output   = f"docking/results/{ligand_id}_out.pdbqt"
    
    if not os.path.exists(ligand):
        return {"ligand_id": ligand_id, "affinity": None, "status": "ligand_missing"}
    
    # Skip empty ligands
    with open(ligand) as f:
        atoms = [l for l in f if l.startswith(("ATOM", "HETATM"))]
    if len(atoms) < 5:
        return {"ligand_id": ligand_id, "affinity": None, "status": "empty_ligand"}
    
    # Delete stale output
    if os.path.exists(output):
        os.remove(output)
    
    cmd = [
        "smina",
        "--receptor", receptor,
        "--ligand", ligand,
        "--center_x", str(center[0]),
        "--center_y", str(center[1]),
        "--center_z", str(center[2]),
        "--size_x", str(box_size),
        "--size_y", str(box_size),
        "--size_z", str(box_size),
        "--exhaustiveness", str(exhaustiveness),
        "--num_modes", str(num_modes),
        "--out", output
    ]
    
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    t1 = time.time()
    
    # ─── Parse affinity (smina uses 'minimizedAffinity') ───
    affinity = None
    if os.path.exists(output):
        with open(output) as f:
            for line in f:
                if "minimizedAffinity" in line:
                    try:
                        affinity = float(line.split()[-1])
                    except (ValueError, IndexError):
                        pass
                    break
                # Fallback for Vina-style output
                elif "VINA RESULT:" in line:
                    try:
                        affinity = float(line.split()[3])
                    except (ValueError, IndexError):
                        pass
                    break
    
    return {
        "ligand_id": ligand_id,
        "affinity": affinity,
        "time_sec": round(t1 - t0, 1),
        "status": "ok" if affinity is not None else "no_affinity"
    }

print("✅ Function 'run_docking_final' defined.")

In [ ]:
# ─── Test on 0KV ───
result = run_docking_final("0KV")
print(f"Ligand: {result['ligand_id']}")
print(f"Affinity: {result['affinity']} kcal/mol")
print(f"Time: {result['time_sec']}s")
print(f"Status: {result['status']}")

In [ ]:
# ─── Re-extract affinities from ALL output files (using smina format) ───
import glob

results_dir = "docking/results"
output_files = sorted(glob.glob(f"{results_dir}/*_out.pdbqt"))

print(f"Processing {len(output_files)} output files...\n")

all_results = []
for f in output_files:
    ligand_id = os.path.basename(f).replace("_out.pdbqt", "")
    
    # Parse affinity
    affinity = None
    with open(f) as fh:
        for line in fh:
            if "minimizedAffinity" in line:
                try:
                    affinity = float(line.split()[-1])
                except (ValueError, IndexError):
                    pass
                break
            elif "VINA RESULT:" in line:
                try:
                    affinity = float(line.split()[3])
                except (ValueError, IndexError):
                    pass
                break
    
    if affinity is not None:
        all_results.append({"ligand_id": ligand_id, "affinity": affinity})

# Build DataFrame
df_all = pd.DataFrame(all_results)
df_all = df_all.sort_values("affinity", ascending=True).reset_index(drop=True)
df_all.insert(0, "rank", range(1, len(df_all) + 1))

# Save
df_all.to_csv(f"{results_dir}/affinities_full.csv", index=False)
print(f"✅ Total recovered: {len(df_all)} ligands")
print(f"✅ Saved: {results_dir}/affinities_full.csv\n")

print("Top 10:")
print(df_all.head(10).to_string(index=False))

## 6.7️⃣ ProLIF Analysis on Top Hits

In [ ]:
# ─── Prepare the top 5 ligands for ProLIF analysis ───
top_5 = df_all.head(5)["ligand_id"].tolist()

print("Top 5 ligands:")
for i, lid in enumerate(top_5, start=1):
    aff = df_all[df_all["ligand_id"] == lid]["affinity"].values[0]
    print(f"   {i}. {lid}   ({aff:.3f} kcal/mol)")

print(f"\nOutput files:")
for lid in top_5:
    path = f"docking/results/{lid}_out.pdbqt"
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"   ✅ {path}  ({size:,} bytes)")
    else:
        print(f"   ❌ {path}  MISSING")

In [ ]:
# ─── Convert the top poses to PDB format for analysis ───
# We need to combine the receptor + docked ligand into a single complex

combined_dir = "docking/results/complexes"
os.makedirs(combined_dir, exist_ok=True)

# Read the receptor (once)
with open("docking/receptor/2zq2_receptor.pdbqt", "r") as f:
    receptor_lines = [l for l in f if l.startswith(("ATOM", "HETATM"))]

print(f"Receptor atoms: {len(receptor_lines)}\n")

# For each top ligand, combine receptor + docked pose
for lid in top_5:
    pose_file = f"docking/results/{lid}_out.pdbqt"
    complex_file = f"{combined_dir}/{lid}_complex.pdb"
    
    # Read the top pose (first MODEL only)
    with open(pose_file, "r") as f:
        ligand_lines = []
        in_first_model = True
        for line in f:
            if line.startswith("MODEL"):
                if ligand_lines:  # Already past first model
                    in_first_model = False
                    break
                continue
            if line.startswith("ENDMDL"):
                break
            if line.startswith(("ATOM", "HETATM")) and in_first_model:
                ligand_lines.append(line)
    
    # Write combined file
    with open(complex_file, "w") as f:
        f.writelines(receptor_lines)
        f.write("TER\n")
        f.writelines(ligand_lines)
        f.write("END\n")
    
    print(f"   ✅ {lid}: {len(ligand_lines)} ligand atoms → {complex_file}")

In [ ]:
# ─── FIX: Convert PDBQT poses to proper PDB using OpenBabel ───
complex_dir = "docking/results/complexes"
os.makedirs(complex_dir, exist_ok=True)

# First, save the receptor as proper PDB (from the original PDB file)
receptor_pdb = "pdb/protein_h.pdb"  # Already proper PDB

print("Converting top 5 poses using OpenBabel...\n")

for lid in top_5:
    pose_pdbqt = f"docking/results/{lid}_out.pdbqt"
    pose_pdb   = f"{complex_dir}/{lid}_ligand.pdb"
    
    # Convert the docked PDBQT pose → proper PDB
    cmd = [
        "obabel",
        "-ipdbqt", pose_pdbqt,
        "-opdb",
        "-O", pose_pdb,
        "-h"  # keep hydrogens
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if os.path.exists(pose_pdb):
        size = os.path.getsize(pose_pdb)
        print(f"   ✅ {lid}: {pose_pdb} ({size:,} bytes)")
    else:
        print(f"   ❌ {lid}: conversion failed")
        print(f"      {result.stderr[:200]}")

print(f"\n✅ Conversion complete.")

In [ ]:
# ─── Combine protein + ligand into complex files ───
protein_pdb = "pdb/protein_h.pdb"

# Read receptor
with open(protein_pdb) as f:
    receptor_lines = [l for l in f if l.startswith(("ATOM", "HETATM"))]

print(f"Receptor atoms: {len(receptor_lines)}\n")

for lid in top_5:
    ligand_pdb = f"{complex_dir}/{lid}_ligand.pdb"
    complex_pdb = f"{complex_dir}/{lid}_complex.pdb"
    
    with open(ligand_pdb) as f:
        ligand_lines = [l for l in f if l.startswith(("ATOM", "HETATM"))]
    
    with open(complex_pdb, "w") as f:
        f.writelines(receptor_lines)
        f.write("TER\n")
        f.writelines(ligand_lines)
        f.write("END\n")
    
    print(f"   ✅ {lid}: {len(ligand_lines)} ligand atoms → {complex_pdb}")

In [ ]:
# ─── ProLIF on the properly converted complex ───
import prolif as plf

lid = top_5[0]
complex_file = f"docking/results/complexes/{lid}_complex.pdb"

u_complex = mda.Universe(complex_file)
protein = u_complex.select_atoms("protein")
ligand = u_complex.select_atoms("not protein")

print(f"Protein atoms: {len(protein)}")
print(f"Ligand atoms:  {len(ligand)}")

# Convert to ProLIF molecules
protein_mol = plf.Molecule.from_mda(protein)
ligand_mol = plf.Molecule.from_mda(ligand)

# Run fingerprint
fp = plf.Fingerprint()
lig_list = [ligand_mol]
fp.run_from_iterable(lig_list, protein_mol)

df_prolif = fp.to_dataframe()
print(f"\n✅ ProLIF analysis complete.")
print(f"   Interaction matrix shape: {df_prolif.shape}")

In [ ]:
# ─── Check distances between protein and ligand ───
lid = top_5[0]
complex_file = f"docking/results/complexes/{lid}_complex.pdb"

u_complex = mda.Universe(complex_file)
protein = u_complex.select_atoms("protein")
ligand = u_complex.select_atoms("not protein")

# Get bounding boxes
prot_min = protein.positions.min(axis=0)
prot_max = protein.positions.max(axis=0)
lig_min = ligand.positions.min(axis=0)
lig_max = ligand.positions.max(axis=0)

print("Protein bounding box:")
print(f"   min: {prot_min}")
print(f"   max: {prot_max}")
print(f"\nLigand bounding box:")
print(f"   min: {lig_min}")
print(f"   max: {lig_max}")

# Compute min distance between any ligand atom and any protein atom
from scipy.spatial import cKDTree
tree = cKDTree(protein.positions)
distances, _ = tree.query(ligand.positions)
min_dist = distances.min()

print(f"\nMinimum distance (protein-ligand): {min_dist:.2f} Å")
print(f"   {'✅ Close enough' if min_dist < 6 else '❌ TOO FAR — coordinate mismatch!'}")

In [ ]:
# ─── Rebuild complexes using ORIGINAL 2zq2 protein ───
import prolif as plf

# Use the ORIGINAL protein (before PDB2PQR shifts)
original_protein = "pdb/2zq2.pdb"

# Read only protein atoms from original
with open(original_protein) as f:
    receptor_lines = [
        l for l in f 
        if l.startswith(("ATOM", "HETATM"))
        and "13U" not in l  # exclude the original ligand
    ]

print(f"Original protein atoms: {len(receptor_lines)}\n")

complex_dir = "docking/results/complexes"
os.makedirs(complex_dir, exist_ok=True)

for lid in top_5:
    ligand_pdb = f"{complex_dir}/{lid}_ligand.pdb"
    complex_pdb = f"{complex_dir}/{lid}_complex_v2.pdb"
    
    with open(ligand_pdb) as f:
        ligand_lines = [l for l in f if l.startswith(("ATOM", "HETATM"))]
    
    with open(complex_pdb, "w") as f:
        f.writelines(receptor_lines)
        f.write("TER\n")
        f.writelines(ligand_lines)
        f.write("END\n")
    
    print(f"   ✅ {lid}: {len(ligand_lines)} ligand atoms")

print(f"\n✅ Complexes rebuilt.")

In [ ]:
# ─── Check distances with the new complex ───
lid = top_5[0]
complex_file = f"{complex_dir}/{lid}_complex_v2.pdb"

u_complex = mda.Universe(complex_file)
protein = u_complex.select_atoms("protein")
ligand = u_complex.select_atoms("not protein")

from scipy.spatial import cKDTree
tree = cKDTree(protein.positions)
distances, _ = tree.query(ligand.positions)

print(f"Ligand atoms: {len(ligand)}")
print(f"Minimum distance: {distances.min():.2f} Å")
print(f"Mean of closest atoms: {distances.mean():.2f} Å")

# Histogram of distances
import numpy as np
print(f"\nDistance distribution:")
print(f"   < 2 Å:   {(distances < 2).sum()} atoms (too close)")
print(f"   2-4 Å:   {((distances >= 2) & (distances < 4)).sum()} atoms (ideal)")
print(f"   4-6 Å:   {((distances >= 4) & (distances < 6)).sum()} atoms (interaction range)")
print(f"   > 6 Å:   {(distances >= 6).sum()} atoms (far)")

In [ ]:
# ─── Reload df_all from the saved CSV ───
import pandas as pd
import os

csv_path = "docking/results/affinities_full.csv"

if os.path.exists(csv_path):
    df_all = pd.read_csv(csv_path)
    print(f"✅ Loaded: {len(df_all)} ligands")
    print(f"\nTop 5:")
    print(df_all.head().to_string(index=False))
    
    # Also restore top_5 list
    top_5 = df_all.head(5)["ligand_id"].tolist()
    print(f"\nTop 5 list: {top_5}")
else:
    print(f"❌ File not found: {csv_path}")

---

### ⚠️ Technical Note — ProLIF on Docked Poses

**Status:** Documented limitation (not a blocker).

**Goal:** Extend ProLIF interaction analysis to the top 5 docked poses.

**6 approaches attempted — all unsuccessful:**

| # | Approach | Outcome |
|---|----------|---------|
| 1 | OpenBabel PDBQT → PDB | 0 interactions |
| 2 | `force=True` in ProLIF | Kernel crash |
| 3 | `plf.pdbqt_supplier` with SDF templates | Kernel crash |
| 4 | Meeko `mk_export.py` | Conversion failure |
| 5 | OpenBabel + `protein_h.pdb` | Coordinate mismatch (1.64 Å) |
| 6 | RDKit `AssignBondOrdersFromTemplate` | Kernel crash |

**Root cause:** Fundamental incompatibility between:
- **smina PDBQT output** — missing hydrogens, no bond orders
- **RDKit converter** (used by ProLIF) — requires explicit hydrogens + valid bond orders

**What works:**
- ProLIF on **crystal structure complexes** (Lesson 14 — verified ✅)

**Workarounds for future work:**
- Use **AutoDock-Vina's Python API** (better PDBQT compatibility)
- Switch to **PLIP** (Protein-Ligand Interaction Profiler)
- Prepare ligands with **Meeko** before docking (not after)

> **Impact:** Docking results (100 valid binding affinities) remain **complete and valid**. The interaction analysis is a **future enhancement**, not a requirement.

---

## 📝 Final Summary — Project Complete

### What We Built

A **complete computational drug discovery pipeline** for Trypsin inhibitors:

1. **Data Collection** — 117 ligands via RCSB Search API (Lesson 13)
2. **Structure Preparation** — receptor hydrogenation + ligand bond fixing (Lesson 14)
3. **Binding Site Analysis** — ProLIF on crystal structure (Lesson 14)
4. **Virtual Screening** — docking of 117 ligands with smina (this lesson)
5. **Ranking** — top 5 hits identified by binding affinity
6. **Visualization** — NGLView + iCN3D

### Final Results

| Metric | Value |
|--------|-------|
| **Total ligands screened** | 117 |
| **Successful dockings** | 100 (85%) |
| **Best binder** | `13U` (-9.54 kcal/mol) |
| **Pipeline validation** | Reference ligand ranked #1 ✅ |

### Top 5 Hits

| Rank | Ligand | Affinity (kcal/mol) |
|------|--------|---------------------|
| 1 | **13U** | **-9.54** |
| 2 | R11 | -9.51 |
| 3 | BAH | -9.40 |
| 4 | 607 | -9.40 |
| 5 | 12U | -9.35 |

### Skills Demonstrated

- **Python** — pandas, numpy, matplotlib
- **Cheminformatics** — RDKit, OpenBabel
- **Structural Biology** — MDAnalysis, Biopython
- **Docking** — smina, binding box design
- **Interaction Analysis** — ProLIF (on crystal structures)
- **Visualization** — NGLView, iCN3D
- **Web APIs** — RCSB PDB, Search API
- **Environment Management** — conda, WSL, Ubuntu

---

## 🚀 Next Steps

1. **Re-run docking** with **AutoDock-Vina Python API** for better compatibility
2. **MD simulations** of top 5 hits (100 ns each with OpenMM)
3. **ADMET predictions** for drug-likeness
4. **Experimental validation** of top hits in lab
5. **Publish** as a reproducible GitHub repository

---

## 🎓 Project Complete

This is the **final lesson** of the MolSSI Python Scripting for Biochemistry series.

**Combined with previous lessons, this project covers:**
- Python fundamentals + data analysis
- Biopython + structural biology
- RCSB Web API + database access
- iCN3D + NGLView visualization
- Ligand discovery (EC classification)
- Structure preparation (ProLIF)
- **Molecular docking (this lesson)**

**Total: 14 Notebooks | 14 Reference Notes | 1 Complete Pipeline**